# The June 2026 heatwave

Météo-France dates the heatwave to 17–30 June 2026, with the hottest days on record in France on 23, 24 and 25 June and storms breaking the heat from 27 June.

This notebook asks two questions:

1. **How early did each model see the peak coming?** Every forecast of a target afternoon, from 10 days ahead down to 12 hours, at the cities at the centre of the records and as a France land average, against both truths.
2. **How did skill change during the event?** Scores for valid times inside 17–30 June against the rest of the summer.

Run first:

```bash
sbatch --gpus-per-node=2 --time=05:00:00 scripts/slurm/season_forecast.sh configs/heatwave2026_france.yaml
uv run scripts/run_season.py configs/heatwave2026_france.yaml --stage score
uv run scripts/run_season.py configs/summer2026_france.yaml --stage score     # for part 2
```

Sites are the 0.25° grid cells nearest each station: a cell is a ~25 km average, so its temperature sits below a station's record maximum (Bordeaux-Mérignac: 42.5 °C on 23 June). With the station truth in the config, the red line is the nearest station's on-the-hour reading.

In [ ]:
# Parameters
CONFIG = "configs/heatwave2026_france.yaml"          # 10-day forecasts around the event
SEASON_CONFIG = "configs/summer2026_france.yaml"      # the whole summer, for part 2
OVERRIDES, SEASON_OVERRIDES = {}, {}
TARGETS = ["2026-06-23T12", "2026-06-24T12", "2026-06-25T12"]   # 12 UTC = 14 h local, the afternoon peak
EVENT = ("2026-06-17T00", "2026-06-30T18")

In [ ]:
import os, sys
from pathlib import Path
root = Path.cwd()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from weather_pipeline.season import config
from weather_pipeline.season.event import convergence
from weather_pipeline.season.scoring import load_statistics, season_scores, valid_time_window

cfg = config.load(CONFIG, **OVERRIDES)
print(cfg.describe())


## 1. How early did the models see it?

Boxes: each model's ensemble (median, quartiles, whiskers to the extreme members) for each lead time, the longest on the left. Lines: the truths at that time. An ensemble that brackets the line long before the event saw it coming; one whose boxes sit below the line until the last day underestimated it.

In [ ]:
conv = {}
for target in TARGETS:
    try:
        conv[target] = convergence(cfg, target)
    except LookupError as e:
        print(e)

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for target, ds in conv.items():
    sites = list(ds["site"].values)
    ncol = 3
    fig, axes = plt.subplots(int(np.ceil(len(sites) / ncol)), ncol, figsize=(15, 4.2 * np.ceil(len(sites) / ncol)),
                             squeeze=False, sharex=True)
    days = ds["lead_time"].values / np.timedelta64(1, "D")
    models = list(ds["model"].values)
    width = 0.8 / len(models) * np.min(np.diff(days)) if len(days) > 1 else 0.3
    for ax, site in zip(axes.flat, sites):
        for k, model in enumerate(models):
            fc = ds["forecast"].sel(model=model, site=site) - 273.15
            data, pos = [], []
            for d, lt in zip(days, ds["lead_time"].values):
                v = fc.sel(lead_time=lt).values
                if np.isfinite(v).any():
                    data.append(v[np.isfinite(v)]); pos.append(-d + (k - (len(models) - 1) / 2) * width)
            if data:
                ax.boxplot(data, positions=pos, widths=width * 0.9, whis=(0, 100), patch_artist=True,
                           boxprops=dict(facecolor=colors[k], alpha=0.45, color=colors[k]),
                           medianprops=dict(color=colors[k]), whiskerprops=dict(color=colors[k]),
                           capprops=dict(color=colors[k]), manage_ticks=False)
                ax.plot([], [], color=colors[k], lw=6, alpha=0.45, label=model)
        for t, style in zip(ds["truth"].values, ("-", "--", ":")):
            v = float(ds["observed"].sel(truth=t, site=site))
            if np.isfinite(v):
                ax.axhline(v - 273.15, color="firebrick" if t == "stations" else "k", ls=style,
                           lw=1.6 if t == "stations" else 1.2, label=t)
        ax.set_title(site); ax.grid(alpha=0.3); ax.set_ylabel("°C")
    for ax in axes.flat[len(sites):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel("days before the target")
    axes.flat[0].legend(fontsize=8, loc="upper left")
    fig.suptitle(f"2 m temperature at {pd.Timestamp(target):%d %B %Y %H} UTC, by forecast lead", fontsize=13)
    fig.tight_layout()


Ensemble-mean error against ERA5 by lead (K; negative = forecast too cool), and whether ERA5 fell inside the ensemble's range (`in` = between the coldest and warmest member).

In [ ]:
rows = []
for target, ds in conv.items():
    obs = ds["observed"].sel(truth=ds["truth"].values[0])
    for model in ds["model"].values:
        for lt in ds["lead_time"].values:
            fc = ds["forecast"].sel(model=model, lead_time=lt)
            if not np.isfinite(fc).any():
                continue
            for site in ds["site"].values:
                m, o = fc.sel(site=site), float(obs.sel(site=site))
                rows.append({"target": target, "model": model, "days ahead": round(lt / np.timedelta64(1, "D"), 1),
                             "site": site, "error": float(m.mean()) - o,
                             "inside": "in" if float(m.min()) <= o <= float(m.max()) else ("below" if float(m.max()) < o else "above")})
table = pd.DataFrame(rows)
if len(table):
    display(table.pivot_table(index=["target", "site", "model"], columns="days ahead", values="error").round(1))
    display(table.pivot_table(index=["target", "site", "model"], columns="days ahead", values="inside", aggfunc="first"))


## 2. Skill inside the event against the rest of the summer

Scores for valid times inside 17–30 June (solid) and outside it (dashed), from the full-summer run.

Read with care. Choosing cases *because* they turned out extreme favours forecasts that over-predict heat (the forecaster's dilemma), so this is a diagnostic of how errors changed, not a fair ranking. The fair ranking comes from scores over all cases with thresholds fixed in advance: the climatology-based extreme scores, next.

In [ ]:
season = config.load(SEASON_CONFIG, **SEASON_OVERRIDES)
fig, axes = plt.subplots(len(season.truths), 3, figsize=(15, 3.8 * len(season.truths)), squeeze=False, sharex=True)
for r, truth in enumerate(season.truths):
    stats = load_statistics(season, truth)
    if not stats:
        print(f"nothing scored against {truth} for {season.name}"); continue
    for k, (model, st) in enumerate(stats.items()):
        for inside, style in ((True, "-"), (False, "--")):
            s = season_scores(valid_time_window(st, *EVENT, inside=inside))
            h = s["lead_time"].values / np.timedelta64(1, "h")
            lab = f"{model}, {'event' if inside else 'rest'}"
            for c, key in enumerate(("crps", "bias", "spread_skill")):
                axes[r, c].plot(h, s[key], style, color=colors[k], marker="o", ms=2.5, label=lab)
    for c, (key, title) in enumerate((("crps", "CRPS (K)"), ("bias", "bias (K)"), ("spread_skill", "spread-skill ratio"))):
        axes[r, c].set_title(f"{title} vs {truth}"); axes[r, c].grid(alpha=0.3)
    axes[r, 1].axhline(0, color="k", lw=0.8); axes[r, 2].axhline(1, color="k", lw=0.8, ls=":")
for ax in axes[-1]:
    ax.set_xlabel("lead time (h)")
axes[0, 0].legend(fontsize=8); fig.tight_layout()


## 3. Records during the event

The extreme scores restricted to valid times inside 17–30 June, from the full-summer run. At quantile 1.0 the threshold is the 1979–2017 ERA5 record for that cell, month and hour, so an event there is a broken record. The miss rate answers the key question directly: when the truth broke the record, how often was it above every ensemble member, i.e. outside anything the model considered possible?

The occurrence ratio and miss rate are conditioned on the event having happened, so they describe how the models behaved in this event; the season-wide extreme scores in `season_report.ipynb` are the fair comparison.

In [ ]:
from weather_pipeline.season.scoring import BASELINE, summed
from weather_pipeline.season import metrics

rows = []
for truth in season.truths:
    for model, st in load_statistics(season, truth).items():
        if not int(st.attrs.get("has_thresholds", 0)):
            print(f"{truth}/{model}: no thresholds yet (run --stage climatology, then --stage score)"); continue
        tot = summed(valid_time_window(st, *EVENT))
        m = metrics.finalise(tot)
        for lt in (np.timedelta64(24, "h"), np.timedelta64(72, "h"), np.timedelta64(120, "h")):
            if lt not in m["lead_time"].values:
                continue
            for q in m["quantile"].values:
                v = m.sel(lead_time=lt, quantile=q)
                rows.append({"truth": truth, "model": model, "lead (h)": int(lt / np.timedelta64(1, "h")),
                             "quantile": q, "event share": float(v.event_frac),
                             "occurrence ratio": float(v.occurrence_ratio), "miss rate": float(v.miss_rate),
                             "Brier": float(v.brier), "twCRPS": float(v.twcrps)})
records = pd.DataFrame(rows)
if len(records):
    display(records[records["quantile"] >= 0.99].pivot_table(
        index=["truth", "quantile", "lead (h)"], columns="model",
        values=["miss rate", "occurrence ratio"]).round(3))
